# pm4py-ucm — save, share & resume a project

The other tutorials turn an event log into **models**, **families**,
**scenarios** and **dashboards**. This one is about *keeping* the analysis:
turning a configured session into a small **project file** you can put down and
pick back up later, or hand to a colleague.

The whole design rests on one distinction:

| Tier | Examples | In a project? |
|------|----------|---------------|
| **Config** — the manual effort | miner settings, CSV mapping, renaming, filters, performers, overlays, decomposition, family & scenario settings, dashboards | **stored** |
| **Log** — the event data | the XES / CSV bytes | **referenced** (settings file) or **bundled** (project file) |
| **Derived** — the outputs | the mined UCM, scenarios, family, reports | **never stored** — recomputed on load |

So a project stores *inputs*; opening it **recomputes** the outputs. That keeps
project files tiny and immune to changes in *how* a model is mined.

The persistence layer lives in **`web/sessions/`** — deliberately
Streamlit-free, so it is plain-data and testable headless (and this notebook
can call it directly). The web app wires `st.session_state` into it; here we
call the same functions by hand.

## 1. Setup

In [1]:
import sys, hashlib, io, json, zipfile
from pathlib import Path

# web/sessions is a package under web/, not part of the installed pm4py_ucm.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "web").is_dir() and (REPO_ROOT.parent / "web").is_dir():
    REPO_ROOT = REPO_ROOT.parent          # running from demo/
sys.path.insert(0, str(REPO_ROOT / "web"))

from sessions import (
    ProjectDoc, LogRef, REGISTRY, collect,
    save_settings, save_bundle, load,
    wrap_registry, unwrap_registry, SCHEMA_VERSION,
)

# A real log, only so the reference (name, kind, hash) is realistic. We never
# mine it here — persistence doesn't care what the bytes are.
DEMO = REPO_ROOT / "demo"
log_bytes = (DEMO / "ClaimsPaymentLog.zip").read_bytes()
log_name = "ClaimsPaymentLog.zip"
# The app identifies a log by the first 16 hex of its SHA-256 (its "file_hash").
file_hash = hashlib.sha256(log_bytes).hexdigest()[:16]
print(f"log: {log_name}  ({len(log_bytes):,} bytes)  file_hash={file_hash}")
print(f"schema version: {SCHEMA_VERSION}")

log: ClaimsPaymentLog.zip  (2,709,054 bytes)  file_hash=1ba8637bddd049e5
schema version: 1


## 2. The parameter registry — one source of truth

The app has *many* settings, and it grows fast. If saving meant "gather the
current settings into a dict" by hand, every new setting a contributor forgot
to add would silently vanish from saved projects. Instead every persistable
setting is declared **once**, in the registry, and a CI test fails if the app's
gather and this list ever drift apart.

In [2]:
import pandas as pd

pd.DataFrame(
    [{"id": p.id, "category": p.category, "default": repr(p.default)}
     for p in REGISTRY]
)

,id,category,default
0,noise_threshold,miner,0.2
1,min_support,miner,0.0
2,notation,miner,'ucm'
3,decomposition,miner,'off'
4,resource_attribute,performers,'org:role'
5,overlay_nodes,overlay,[]
6,overlay_edges,overlay,[]
7,filter_spec,transform,[]
8,csv_columns,csv,None
9,scenario_strategy,scenarios,'variant'


`collect()` serialises a `{id: value}` gather into the project's `config`
block, and it **enforces the contract**: passing an unregistered id — or
*omitting* a registered one — raises, so "what the app knows" and "what we
persist" can't silently diverge. The easiest way to build a valid gather is to
start from the registry defaults and override what you changed:

In [3]:
values = {p.id: p.default for p in REGISTRY}   # every id, at its default
values["noise_threshold"] = 0.3
values["notation"] = "bpmn"
values["filter_spec"] = [["exclude_activities", ["Cancel Claim"]]]
values["family_attrs"] = ["Country"]
values["active_view"] = "Dashboards"

config = collect(values)
print(json.dumps(config, indent=2)[:500], "...")

{
  "noise_threshold": 0.3,
  "min_support": 0.0,
  "notation": "bpmn",
  "decomposition": "off",
  "resource_attribute": "org:role",
  "overlay_nodes": [],
  "overlay_edges": [],
  "filter_spec": [
    [
      "exclude_activities",
      [
        "Cancel Claim"
      ]
    ]
  ],
  "csv_columns": null,
  "scenario_strategy": "variant",
  "scenario_group_name": "MinedScenarios",
  "scenario_max_loop_iterations": 2,
  "scenario_decision_tree_max_depth": 3,
  "family_attrs": [
    "Country"
  ],
 ...


## 3. A project document, and the settings file

A `ProjectDoc` ties together a **log reference**, the **config**, and
(optionally) the **dashboards**. `save_settings` serialises it to a small JSON
file — configuration only, **no event data** — which is the privacy-preserving
way to share when the log is sensitive.

In [4]:
doc = ProjectDoc(
    log=LogRef(source="sample", name=log_name, kind="zip", sha256=file_hash),
    config=config,
    app_version="0.7.1",
)

settings_bytes = save_settings(doc)
print(f"{len(settings_bytes):,} bytes  ->  <log>.ucmproj.json\n")
print(settings_bytes.decode("utf-8")[:420], "...")

1,014 bytes  ->  <log>.ucmproj.json

{
  "format": "pm4py-ucm-project",
  "schema_version": 1,
  "app_version": "0.7.1",
  "created_utc": "",
  "log": {
    "source": "sample",
    "name": "ClaimsPaymentLog.zip",
    "kind": "zip",
    "sha256": "1ba8637bddd049e5"
  },
  "config": {
    "noise_threshold": 0.3,
    "min_support": 0.0,
    "notation": "bpmn",
    "decomposition": "off",
    "resource_attribute": "org:role",
    "overlay_nodes": [],
    "o ...


## 4. Dashboards travel too

Dashboards are the one setting that doesn't live in `st.session_state` — they
live in the browser island's `localStorage`. A small versioned **bridge** reads
them back on save and restores them on resume; here we just show the shape it
stores. `wrap_registry` puts the island's dashboard registry into a versioned
envelope; `unwrap_registry` is its tolerant inverse.

In [5]:
# What the bridge reads out of the browser: a registry of named dashboards.
island_registry = {
    "active": "d1",
    "dashboards": [
        {"id": "d1", "name": "Ops overview",
         "specs": [{"id": "w1", "metric": "duration", "viz": "kpi"}],
         "filters": []},
        {"id": "d2", "name": "SLA watch", "specs": [], "filters": []},
    ],
}

doc.dashboards = wrap_registry(island_registry)     # -> {"version": N, "registry": {...}}
print("stored envelope:", json.dumps(doc.dashboards)[:120], "...")

# Tolerant inverse — accepts the envelope (and, as a fallback, a bare registry).
restored = unwrap_registry(doc.dashboards)
print("round-trips     :", restored == island_registry)
print("dashboards      :", [d["name"] for d in restored["dashboards"]])

# Re-serialise now that the dashboards are attached, so the settings file below
# carries them too.
settings_bytes = save_settings(doc)

stored envelope: {"version": 1, "registry": {"active": "d1", "dashboards": [{"id": "d1", "name": "Ops overview", "specs": [{"id": "w1", " ...
round-trips     : True
dashboards      : ['Ops overview', 'SLA watch']


## 5. The project bundle — config + log in one file

`save_bundle` produces a self-contained zip: the same `project.json` plus the
event log alongside. It is one-click to resume or share — but it *does* ship the
data, so it's the choice when you want the recipient to have everything.

In [6]:
bundle_bytes = save_bundle(doc, log_name, log_bytes)
print(f"{len(bundle_bytes):,} bytes  ->  <log>.ucmproj.zip   (starts with {bundle_bytes[:2]!r} = a zip)\n")

with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as zf:
    for info in zf.infolist():
        print(f"  {info.filename:<28} {info.file_size:>10,} bytes")

2,703,578 bytes  ->  <log>.ucmproj.zip   (starts with b'PK' = a zip)

  project.json                      1,503 bytes
  log/ClaimsPaymentLog.zip      2,709,054 bytes


## 6. Resume — `load()` round-trips either file

`load()` auto-detects a settings file vs a bundle (by the zip magic) and
returns `(doc, log)`: `log` is `(name, bytes)` for a bundle, or `None` for a
settings file (the app then re-supplies the log, matching it by `sha256`).

In [7]:
# Settings file: no log inside.
doc_s, log_s = load(settings_bytes)
print("settings ->", "log:", log_s, "| notation:", doc_s.config["notation"],
      "| dashboards:", [d["name"] for d in unwrap_registry(doc_s.dashboards)["dashboards"]])

# Bundle: brings its own log.
doc_b, log_b = load(bundle_bytes)
name, data = log_b
print("bundle   ->", f"log: ({name}, {len(data):,} bytes)  matches:",
      hashlib.sha256(data).hexdigest()[:16] == doc_b.log.sha256)

# Config survives the round-trip exactly.
print("config identical:", doc_s.config == config)

settings -> log: None | notation: bpmn | dashboards: ['Ops overview', 'SLA watch']
bundle   -> log: (ClaimsPaymentLog.zip, 2,709,054 bytes)  matches: True
config identical: True


## 7. Future-proofing — unknown keys are preserved

A project written by a *newer* app (with a setting this version doesn't know)
must still round-trip through an *older* one without losing that setting.
Unknown top-level and unknown `config` keys are carried through untouched, and
missing keys fall back to the registry default. Together with a versioned
`schema_version` (and a migration chain), that's what lets any app version open
any project it understands.

In [8]:
raw = json.loads(save_settings(doc).decode("utf-8"))
raw["config"]["a_future_setting"] = 42          # from a newer app
raw["a_future_top_level_block"] = {"x": 1}

reloaded = ProjectDoc.from_dict(raw)
out = reloaded.to_dict()
print("unknown config key kept :", out["config"].get("a_future_setting"))
print("unknown top-level kept  :", out.get("a_future_top_level_block"))

unknown config key kept : 42
unknown top-level kept  : {'x': 1}


## 8. In the web app

Everything above is point-and-click in **`web/streamlit_app_v5.py`**:

* the sidebar's **Project** group offers **⬇ Save settings** and **⬇ Save
  project bundle** (with a caption showing how many dashboards travel with it);
* **↻ Resume a saved project** in the log-source area loads either file, re-seeds
  every widget, re-attaches or re-requests the log, and reports anything only
  partially applied.

Because only *inputs* are stored, opening a project **recomputes** the outputs:
the Model and Dashboards views recompute as you view them, and — since the
Family view mines on demand — opening the **Family** tab after a resume
re-mines the family automatically once, so **Compare** (which reads the family)
follows. Nothing derived is ever trusted from the file.

## 9. Wrap-up

```python
import sys; sys.path.insert(0, "web")
from sessions import (ProjectDoc, LogRef, REGISTRY, collect,
                      save_settings, save_bundle, load,
                      wrap_registry, unwrap_registry)

values = {p.id: p.default for p in REGISTRY}     # start from defaults
values["noise_threshold"] = 0.3                  # override what changed
doc = ProjectDoc(log=LogRef("upload", "log.xes", "xes", file_hash),
                 config=collect(values),
                 dashboards=wrap_registry(island_registry))
save_settings(doc)                 # -> <log>.ucmproj.json  (config only)
save_bundle(doc, "log.xes", log_bytes)   # -> <log>.ucmproj.zip  (config + log)
doc2, log = load(file_bytes)       # auto-detects settings vs bundle
```

Where to go next:

* [`docs/sessions.md`](../docs/sessions.md) — the full design: the registry, the
  CI drift guard, the schema/migrations, and the dashboards bridge.
* The other tutorials — `pm4py_ucm_tutorial.ipynb`,
  `model_families_tutorial.ipynb`, `scenario_synthesis_tutorial.ipynb`,
  `dashboards_tutorial.ipynb` — produce the very outputs a project reproduces.